# Gold Layer - Trending Products Analysis

## Objective

The objective of this notebook is to identify products that are currently trending by comparing their order volume between the latest month and the previous month.

Unlike Product Ranking, which measures overall performance, this notebook measures product momentum.

This Gold table directly answers the RetailMart business question:

**"Which products are trending?"**

The resulting table is optimized for executive dashboards, business reporting, and product performance analysis.

In [0]:
%run ../00_Setup/01_Config

In [0]:
%run ../Utils/Common_Utils

In [0]:
SOURCE = GOLD_FACT_SALES
TARGET_TABLE = GOLD_TRENDING_PRODUCT

In [0]:
## Step 1: Build Trending Products Table
spark.sql(f"""

CREATE OR REPLACE TABLE {TARGET_TABLE}
USING DELTA
AS
WITH monthly_product_orders AS (

    SELECT
        product_id,
        product_category_name,
        order_year,
        order_month,
        COUNT(DISTINCT order_id) AS monthly_orders,
        SUM(total_payment_value) AS monthly_revenue
    FROM {GOLD_FACT_SALES}
    WHERE order_status = 'delivered'
    GROUP BY
        product_id,
        product_category_name,
        order_year,
        order_month
),
ranked_months AS (
    SELECT
        *,
        DENSE_RANK() OVER (
            PARTITION BY product_id
            ORDER BY order_year DESC, order_month DESC
        ) AS month_rank
    FROM monthly_product_orders
),
latest_vs_previous AS (
    SELECT
        product_id,
        product_category_name,
        MAX(CASE WHEN month_rank = 1 THEN monthly_orders ELSE 0 END) AS latest_month_orders,

        MAX(CASE WHEN month_rank = 2 THEN monthly_orders ELSE 0 END) AS previous_month_orders,

        MAX(CASE WHEN month_rank = 1 THEN monthly_revenue ELSE 0 END) AS latest_month_revenue,

        MAX(CASE WHEN month_rank = 2 THEN monthly_revenue ELSE 0 END) AS previous_month_revenue
    FROM ranked_months
    WHERE month_rank <= 2
    GROUP BY
        product_id,
        product_category_name
)
SELECT
    product_id,
    product_category_name,
    latest_month_orders,
    previous_month_orders,
    latest_month_revenue,
    previous_month_revenue,
    (latest_month_orders - previous_month_orders) AS order_growth,
    ROUND(
        CASE
            WHEN previous_month_orders = 0 THEN NULL
            ELSE
                (latest_month_orders - previous_month_orders)
                * 100.0
                / previous_month_orders
        END,
        2
    ) AS growth_percentage
FROM latest_vs_previous
WHERE latest_month_orders > 0
ORDER BY
    order_growth DESC,
    growth_percentage DESC
""")
print("Gold Trending Products table created successfully.")

Gold Trending Products table created successfully.


In [0]:
# Validate Gold Table
trending_products_df = spark.table(TARGET_TABLE)
display(trending_products_df.limit(10))

product_id,product_category_name,latest_month_orders,previous_month_orders,latest_month_revenue,previous_month_revenue,order_growth,growth_percentage
PROD_001504,music,7,1,7218.629999999999,1912.58,6,600.00
PROD_000908,toys,5,1,4011.61,608.92,4,400.00
PROD_000324,health,4,1,6149.41,87.95,3,300.00
PROD_000517,home_appliances,4,1,4889.23,365.63,3,300.00
PROD_000679,garden,4,1,6644.879999999999,389.39,3,300.00
PROD_001069,beauty,4,1,5068.099999999999,1243.3,3,300.00
PROD_001513,computers,4,1,3027.11,1240.41,3,300.00
PROD_002235,toys,4,1,7581.12,221.16,3,300.00
PROD_002285,home_appliances,4,1,7365.38,461.78,3,300.00
PROD_000077,garden,3,1,7226.12,2951.45,2,200.00


In [0]:
trending_products_df.describe().show()

+-------+-----------+---------------------+-------------------+---------------------+--------------------+----------------------+------------------+-----------------+
|summary| product_id|product_category_name|latest_month_orders|previous_month_orders|latest_month_revenue|previous_month_revenue|      order_growth|growth_percentage|
+-------+-----------+---------------------+-------------------+---------------------+--------------------+----------------------+------------------+-----------------+
|  count|       3000|                 3000|               3000|                 3000|                3000|                  3000|              3000|             3000|
|   mean|       NULL|                 NULL| 1.2386666666666666|   1.2736666666666667|  1923.2477266666708|    1917.2757233333293|            -0.035|         8.971587|
| stddev|       NULL|                 NULL|   0.51555163341592|   0.5436344375492174|  1250.5052263831217|    1252.2090858849413|0.7562841101189448|55.21351246330229

In [0]:
## Step 3: Top 20 Trending Products
display(

spark.sql(f"""

SELECT

    product_id,

    product_category_name,

    latest_month_orders,

    previous_month_orders,

    order_growth,

    growth_percentage

FROM {TARGET_TABLE}

ORDER BY

    order_growth DESC,

    growth_percentage DESC

LIMIT 20

""")

)

product_id,product_category_name,latest_month_orders,previous_month_orders,order_growth,growth_percentage
PROD_001504,music,7,1,6,600.00
PROD_000908,toys,5,1,4,400.00
PROD_002235,toys,4,1,3,300.00
PROD_002285,home_appliances,4,1,3,300.00
PROD_001069,beauty,4,1,3,300.00
PROD_000324,health,4,1,3,300.00
PROD_001513,computers,4,1,3,300.00
PROD_000517,home_appliances,4,1,3,300.00
PROD_000679,garden,4,1,3,300.00
PROD_000128,food,3,1,2,200.00


In [0]:
# Category wise trending product
display(

spark.sql(f"""

SELECT

    product_category_name,

    COUNT(*) AS trending_products,

    ROUND(AVG(growth_percentage),2) AS average_growth_percentage

FROM {TARGET_TABLE}

GROUP BY

    product_category_name

ORDER BY

    average_growth_percentage DESC

""")

)

product_category_name,trending_products,average_growth_percentage
toys,208,18.07
home_appliances,185,15.45
music,191,13.09
beauty,193,12.52
office,187,11.45
garden,233,10.44
sports,180,9.86
books,215,9.22
computers,188,7.09
electronics,199,6.20


In [0]:
# Gold Table summary
print(f"Target Table : {TARGET_TABLE}")

print(f"Total Products : {trending_products_df.count()}")

print(f"Columns : {len(trending_products_df.columns)}")

Target Table : retailmart.gold.trending_product
Total Products : 3000
Columns : 8


# Engineering Observations

- Built a Gold Layer table for identifying trending products.
- Used delivered orders to measure actual product demand.
- Compared product performance between the latest month and the previous month.
- Applied the DENSE_RANK() window function to identify the two most recent months for each product.
- Calculated order growth and month-over-month growth percentage.
- This Gold table directly answers the RetailMart business question:
  **"Which products are trending?"**
- The output is optimized for executive dashboards and product performance reporting.